# Capítulo 4 — Introdução à Regressão Linear

Companion em Python inspirado na sequência conceitual do Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander.

> Objetivo: implementar os conceitos matemáticos e financeiros, não reproduzir o texto do livro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log, exp, sqrt

from quantfinance.regression import ols_coefficients

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
import statsmodels.api as sm
import statsmodels.stats.api as sms

## I.4.2 — Regressão linear simples e OLS

In [ ]:
rng = np.random.default_rng(42)
x = rng.normal(0, 1, 300)
eps = rng.normal(0, 0.8, 300)
y = 0.5 + 1.3*x + eps

X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
print(model.summary())

### OLS manual em álgebra matricial

In [ ]:
beta_hat = ols_coefficients(X, y)
print("beta OLS manual:", beta_hat)
print("beta statsmodels:", model.params)

## I.4.2.4 — ANOVA, R² e resíduos

In [ ]:
resid = model.resid
print("R²:", model.rsquared)
print("RSS:", np.sum(resid**2))
print("TSS:", np.sum((y-y.mean())**2))

## I.4.4 — Regressão múltipla

In [ ]:
x1 = rng.normal(size=500)
x2 = 0.7*x1 + rng.normal(scale=0.7, size=500)
x3 = rng.normal(size=500)
y = 0.2 + 0.8*x1 - 0.4*x2 + 0.6*x3 + rng.normal(scale=0.8, size=500)

X = pd.DataFrame({"x1":x1, "x2":x2, "x3":x3})
Xc = sm.add_constant(X)
multi = sm.OLS(y, Xc).fit()
print(multi.summary())

## I.4.4.8 — Multicolinearidade

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame({
    "variável": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
display(vif)

## I.4.5 — Autocorrelação e heterocedasticidade

In [ ]:
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_white, acorr_breusch_godfrey

print("Durbin-Watson:", durbin_watson(multi.resid))

white = het_white(multi.resid, multi.model.exog)
print("White LM stat / p-value:", white[0], white[1])

bg = acorr_breusch_godfrey(multi, nlags=4)
print("Breusch-Godfrey LM stat / p-value:", bg[0], bg[1])

## I.4.5.5 — GLS/WLS

In [ ]:
# Exemplo WLS em presença de variância conhecida crescente
n = 400
x = np.linspace(0, 10, n)
sigma = 0.5 + 0.2*x
y = 1 + 2*x + rng.normal(scale=sigma)

X = sm.add_constant(x)
ols = sm.OLS(y, X).fit()
wls = sm.WLS(y, X, weights=1/sigma**2).fit()

print("OLS :", ols.params)
print("WLS :", wls.params)

## I.4.6 — Aplicações em finanças: beta, alpha e hedge ratio

In [ ]:
# CAPM sintético
market = rng.normal(0.0004, 0.012, 1000)
asset = 0.0001 + 1.25*market + rng.normal(0, 0.008, 1000)

capm = sm.OLS(asset, sm.add_constant(market)).fit()
alpha, beta = capm.params

print("alpha:", alpha)
print("beta :", beta)

# hedge ratio mínimo-variância via regressão
futures = rng.normal(0, 0.01, 1000)
spot = 0.85*futures + rng.normal(0, 0.006, 1000)
hedge = sm.OLS(spot, sm.add_constant(futures)).fit()
print("Hedge ratio estimado:", hedge.params[1])